In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets transformers accelerate soundfile librosa evaluate jiwer tensorboard gradio

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
from datasets import load_dataset, DatasetDict, Audio
from transformers import WhisperFeatureExtractor
from transformers import WhisperProcessor
from transformers import WhisperTokenizer
from transformers import WhisperProcessor


In [ ]:
horoscop = DatasetDict()

horoscop["train"] = load_dataset("iulik-pisik/horoscop_neti_mijloc", split='train+validation', trust_remote_code=True)
horoscop["test"] = load_dataset("iulik-pisik/horoscop_neti_mijloc", split='test', trust_remote_code=True)


Generating train split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 0it [00:00, ?it/s]
Se citesc datele...: 1276it [00:00, 6564.89it/s]


Generating test split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 184it [00:00, 6730.20it/s]


Generating validation split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 364it [00:00, 7716.63it/s]


In [ ]:
horoscop

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'sentence'],
        num_rows: 1640
    })
    test: Dataset({
        features: ['path', 'audio', 'sentence'],
        num_rows: 184
    })
})

In [ ]:
horoscop["train"][0]

{'path': '/root/.cache/huggingface/datasets/downloads/extracted/45a888a277c362e277e6dcddaee4e308f87f9e718bb5cc6de7244917b14b7995/100_0.wav',
 'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/45a888a277c362e277e6dcddaee4e308f87f9e718bb5cc6de7244917b14b7995/100_0.wav',
  'array': array([ 1.22473466e-07, -1.75164445e-07,  2.20731465e-07, ...,
          4.84011043e-03,  3.83009319e-03,  3.98020353e-03]),
  'sampling_rate': 16000},
 'sentence': 'Bună dimineața! Cod QR de scanat cu telefonul mobil ca să vedem ce se întâmplă la horoscop. E o zi cu energie.'}

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="Romanian", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="Romanian", task="transcribe")


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
horoscop = horoscop.map(prepare_dataset, remove_columns=horoscop.column_names["train"], num_proc=4)


Map (num_proc=4):   0%|          | 0/1640 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/184 [00:00<?, ? examples/s]

In [ ]:
horoscop

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1640
    })
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 184
    })
})

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
import evaluate

metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
model.generation_config.language = "ro"  # define your language of choice here


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./horoscope_model_base",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=horoscop["train"],
    eval_dataset=horoscop["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
1000,0.014200,0.270651,18.033585
2000,0.001200,0.307255,17.060112
3000,0.000700,0.322478,17.011438
4000,0.000500,0.328166,17.035775


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/do

TrainOutput(global_step=4000, training_loss=0.07420636825193651, metrics={'train_runtime': 9511.1386, 'train_samples_per_second': 6.729, 'train_steps_per_second': 0.421, 'total_flos': 4.13132231540736e+18, 'train_loss': 0.07420636825193651, 'epoch': 38.83})

In [ ]:
kwargs = {
    "dataset_tags": "iulik-pisik/horoscop_neti",
    "dataset": "Horoscop Neti",
    "dataset_args": "config: ro, split: test",
    "language": "ro",
    "model_name": "Whisper Base Romanian - Horoscop Neti",
    "finetuned_from": "openai/whisper-base",
    "tasks": "automatic-speech-recognition",
    "tags": "hf-asr-leaderboard",
}


In [ ]:
trainer.push_to_hub(**kwargs)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


events.out.tfevents.1708795521.ed5eb21aad9a.1164.0:   0%|          | 0.00/40.8k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/iulik-pisik/horoscope_model_base/commit/7d2561a7f31b2169cca22ca65c9c7fa17073451f', commit_message='End of training', commit_description='', oid='7d2561a7f31b2169cca22ca65c9c7fa17073451f', pr_url=None, pr_revision=None, pr_num=None)